# LC 104 — Maximum Depth of Binary Tree
**Difficulty:** Easy &nbsp;|&nbsp; **Category:** Trees / DFS
**Pattern:** Recursive DFS — Post-order

<div style="border-left:4px solid purple; padding:10px 16px;
            background:#f5f0ff; margin-top:12px;">
<strong>Core Insight:</strong> The depth of any node is
1 + the deeper of its two children. Ask each subtree
for its depth, take the max, add 1 — recursion handles
the rest.
</div>

## Official Problem Statement

Given the `root` of a binary tree, return its
maximum depth.

A binary tree's maximum depth is the number of nodes
along the longest path from the root node down to
the farthest leaf node.

**Example 1:**
```
    3
   / \
  9  20
    /  \
   15   7
```
```
Input: root = [3,9,20,null,null,15,7]
Output: 3
```
**Example 2:**
```
Input: root = [1,null,2]
Output: 2
```

**Constraints:**
- `0 <= number of nodes <= 10^4`
- `-100 <= Node.val <= 100`

## What This Is Actually Asking

How many floors does the tallest branch of this tree
have?
Count from the root (floor 1) down to the deepest
leaf.
A leaf is any node with no children.
An empty tree has depth 0.

## Walk Through an Example by Hand

```
Tree:       3
           / \
          9  20
            /  \
           15   7

maxDepth(3):
  left  = maxDepth(9)
    left  = maxDepth(None) -> 0
    right = maxDepth(None) -> 0
    return 1 + max(0, 0) = 1
  right = maxDepth(20)
    left  = maxDepth(15)
      return 1 + max(0,0) = 1
    right = maxDepth(7)
      return 1 + max(0,0) = 1
    return 1 + max(1, 1) = 2
  return 1 + max(1, 2) = 3

Answer: 3
```

## The Picture

```
Each node asks its children: "How deep are you?"
Then it adds 1 for itself.

        3          <- returns 3 (1 + max(1,2))
       / \
      9  20        <- 9 returns 1,  20 returns 2
        /  \
       15   7      <- both return 1

Leaf nodes return 1.
None returns 0.
Every other node returns 1 + max(left, right).

The answer bubbles UP from the leaves.
```

## When To Use This Pattern

- When you see **"depth", "height", "levels"** in a tree,
  think **recursive DFS — answer bubbles up from leaves**
- When the problem reduces to **one question per node**,
  think **recursion handles the rest**
- When the base case is **None node**, think **return 0**
- When you need **level-by-level depth**, think
  **BFS counts levels naturally** as an alternative

## The Approach

If the node is None, return 0 — that is the base case.
Otherwise ask the left subtree for its depth, ask the
right subtree for its depth, take the larger of the
two, and add 1 for the current node.
Return that value up to the caller.

In [ ]:
from collections import deque  # needed for BFS alternative
from typing import Optional    # type hints

In [ ]:
# Tree node used by all tree problems
class TreeNode:
    def __init__(self, val=0, left=None, right=None):
        self.val = val
        self.left = left
        self.right = right


def build_tree(vals: list) -> Optional[TreeNode]:
    """Build tree from level-order list (None = missing)."""
    if not vals or vals[0] is None:
        return None
    root = TreeNode(vals[0])
    queue = deque([root])
    i = 1
    while queue and i < len(vals):
        node = queue.popleft()
        if i < len(vals) and vals[i] is not None:
            node.left = TreeNode(vals[i])
            queue.append(node.left)
        i += 1
        if i < len(vals) and vals[i] is not None:
            node.right = TreeNode(vals[i])
            queue.append(node.right)
        i += 1
    return root


def test_harness(func):
    tests = [
        # (level-order list, expected depth)
        ([3,9,20,None,None,15,7],  3),  # example 1
        ([1,None,2],               2),  # right-only chain
        ([],                       0),  # empty tree
        ([1],                      1),  # single node
        ([1,2,3,4,5],              3),  # full tree
        ([1,2,None,3,None,4],      4),  # left-skewed
        ([1,None,2,None,3],        3),  # right-skewed
    ]

    passed = 0
    for i, (vals, expected) in enumerate(tests):
        root = build_tree(vals)
        result = func(root)
        ok = result == expected
        status = "PASSED" if ok else "FAILED"
        if ok:
            passed += 1
        print(
            f"Test {i+1}: {status} | tree={vals} | "
            f"expected={expected} | got={result}"
        )

    print(f"\n{passed}/{len(tests)} tests passed")

In [ ]:
def maxDepth(root: Optional[TreeNode]) -> int:
    """
    Return the maximum depth of a binary tree.

    Base case: None node returns 0. Otherwise recursively
    get depth of left and right subtrees. Return
    1 + max(left_depth, right_depth). The answer
    bubbles up from the leaves to the root.

    Time:  O(n) — every node visited exactly once
    Space: O(h) — call stack depth = tree height
                  O(log n) balanced, O(n) skewed
    """
    pass


# Quick debug — run this cell while building
t1 = build_tree([3,9,20,None,None,15,7])
t2 = build_tree([1,None,2])
t3 = build_tree([])
t4 = build_tree([1])
print(maxDepth(t1))  # expected: 3
print(maxDepth(t2))  # expected: 2
print(maxDepth(t3))  # expected: 0
print(maxDepth(t4))  # expected: 1

In [ ]:
# Uncomment and run when solution is ready
# test_harness(maxDepth)

## Complexity

| Approach | Time | Space |
|---|---|---|
| Brute force — measure every path | O(n log n) | O(h) |
| Recursive DFS | O(n) | O(h) |
| Iterative BFS (level count) | O(n) | O(w) |

DFS and BFS are both O(n) — choose BFS when the tree
may be heavily skewed to avoid deep call stacks.

## Real World Connection

At Citi, the server dependency graph is a tree —
root services depend on middleware, which depend on
database and storage layers.
Maximum depth tells the capacity team the longest
chain of failures if the root degrades: a 5-level
deep tree means a root incident can cascade through
5 tiers before hitting end users.
The same recursive depth pattern runs in the AWS
migration dependency analyser — it walks the service
dependency tree to find the maximum migration blast
radius before scheduling any cutover.

> **Simplicity and clarity is Gold.** — Sean's Study Mantra